In [2]:
library(tidyverse)
library(lubridate)

In [5]:
# State Crosswalk 
state_xwalk <- tibble(
  state_abb  = state.abb,
  state_name = state.name
)

# Load and Clean KFF Data 
kff_data <- read_csv("../data/KFF.csv", skip = 2) %>%
  rename(state_name = 1) %>% 
  filter(!is.na(state_name))

# Load HCRIS Data 
hcris_raw <- read_delim("../data/HCRIS_Data_v2010.txt", delim = "\t")

# Process and Merge Data 
final_dataset <- hcris_raw %>%
  mutate(
    uncomp_care = tot_uncomp_care_charges - tot_uncomp_care_partial_pmts + bad_debt,
    fy_end         = mdy(fy_end),
    fy_start       = mdy(fy_start),
    date_processed = mdy(date_processed),
    date_created   = mdy(date_created),    
    tot_discounts  = abs(tot_discounts),
    hrrp_payment   = abs(hrrp_payment),    
    fyear = year(fy_end)
  ) %>%
  filter(fyear >= 2010 & fyear <= 2021) %>%
  
  left_join(state_xwalk, by = c("state" = "state_abb")) %>%
  
  left_join(kff_data, by = "state_name") %>%
  
  filter(!state %in% c("DC", "GU", "PR", "MP", "VI")) %>%
  
  select(-any_of("year")) %>%
  
  arrange(provider_number, fyear)

# Save the Final Dataset ---
write_csv(final_dataset, "../data/Final_HCRIS_KFF.csv")

Rows: 52 Columns: 3
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): Location, Status of Medicaid Expansion Decision, Expansion Implemen...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 63852 Columns: 48
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): provider_number, fy_start, fy_end, date_processed, date_created, d...
dbl (35): report, status, year, beds, tot_charges, tot_discounts, net_pat_re...
lgl  (1): npi

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [6]:
# Quick Summary Check
print(paste("Final dataset contains", nrow(final_dataset), "rows."))
print(table(final_dataset$fyear))

[1] "Final dataset contains 63057 rows."

2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 
5785 6153 6083 6093 6077 6110 6099 6080 6067 5988 2522 
